# MIL-CREDA bajo ruido de etiqueta — la curva de degradación

Este cuaderno mide una sola cosa: **cuánto pierde cada método a medida que el
material de entrenamiento se contamina**, y si el orden entre métodos sobrevive.

La contaminación reemplaza parte de cada bolsa de entrenamiento por imágenes de
otra clase y **no toca la etiqueta de la bolsa**. Las bolsas son puras y ninguna
instancia lleva etiqueta propia, así que no hay nada que dar vuelta: lo que se
corrompe es la evidencia que una bolsa ofrece a favor de su propia etiqueta.

Esa misma sustitución es **dos perturbaciones distintas**, y esa asimetría es el
experimento y no un detalle. El cableado difunde la etiqueta de la bolsa a sus
treinta instancias, así que un brazo de unidad-instancia recibe instancias
genuinamente mal etiquetadas; un brazo de unidad-bolsa mantiene la etiqueta en la
bolsa y ve testigos que su atención puede aprender a despesar. Leída como una
sola perturbación, la tabla diría «mismo ruido, distinta robustez» cuando dice
«una contaminación, dos consecuencias que las formulaciones implican».

> **Sobre qué transferencia, y por qué esa.**
>
> Una sola, y elegida por la menor brecha de dominio — nunca por cuál dio mejor
> resultado. La regla es del instrumento: una transferencia que ya está cerca de
> su piso con ρ=0 no tiene de dónde caer y no puede mostrar una curva. La brecha
> es propiedad del material y no de ninguna medición, así que nada de lo que
> salga de acá pudo haberla elegido.
>
> **Los roles de selección y evaluación están limpios en todas las tasas.** El
> ruido vive en el material de entrenamiento y en ningún otro lado: `valid` es
> donde la búsqueda lee su criterio y `eval` es la clave de corrección del
> veredicto. Se entrena sucio y se mide limpio, así que el acierto siempre se
> mide contra las etiquetas verdaderas.
>
> **Y los techos se buscaron en limpio.** La curva es el coeficiente elegido sin
> contaminación aplicado con ella, que además es la situación práctica. Lo que
> eso cuesta es que una caída no se puede atribuir: el término fallando y el
> coeficiente quedándose corto se ven igual. Ese es exactamente el trabajo del
> cuaderno de diagnóstico, y no de este.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

# Read AND write: this notebook writes into config.RESULTS too (the
# curve PDFs below, report.txt/report.md, the runs it exports), so a
# read-only override would render a scratch merge and then write its
# report into the canonical Results directory -- exactly the copy
# tools/bridge.py's own provenance note forbids. Mirrors MIL_CREDA_REPO
# above: unset means "behave exactly as before".
MIL_CREDA_RESULTS_OVERRIDE = os.environ.get("MIL_CREDA_RESULTS")

In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, contamination, figures, tables


def show(text: str) -> None:
    display(Markdown(text))


TRANSFERENCIA = "{}->{}".format(*config.NOISE_TRANSFER)
corridos, faltan = contamination.available(), contamination.missing()

print(f"transferencia: {TRANSFERENCIA}")
print(f"niveles declarados: {[f'{r:g}' for r in config.NOISE_LEVELS]}")
print(f"con registro:       {[f'{r:g}' for r in corridos]}")
if faltan:
    print(f"todavía sin correr: {[f'{r:g}' for r in faltan]}")

# Un directorio no es evidencia. La tasa que gobierna una tabla es la que la
# campaña selló en sus propias cotas, y si las dos no coinciden el que está mal
# es el directorio -- así que la discrepancia se dice, no se resuelve a favor de
# la que resulte más cómoda.
for tasa in corridos:
    nivel = contamination.load(tasa)
    if contamination.mismatched(nivel):
        print(f"  ATENCIÓN ρ={tasa:g}: el registro dice "
              f"labelNoise={contamination.stated_rate(nivel)}")

## 1 · Cuánto cae el acierto en destino

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_noise("targetAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_noise("targetAccuracy"))

### 1a · La misma caída, dibujada

La figura califica y la tabla cuantifica, en ese orden: la forma primero, los
números después. Se muestra acá adentro **y** se guarda en vectorial — la copia
archivada queda en `Results/Noise`, y ninguna de las dos reemplaza a la otra.

In [ ]:
figura = figures.noise_curves("targetAccuracy",
                              config.PRODUCT / "Results" / "Noise" / "degradation")
figures.inline(figura)

## 2 · Qué le pasa a la participación del término de adaptación

In [ ]:
show(tables.objective("noise.share"))

In [ ]:
show(tables.render_noise("adaptationShare", markdown=True))

In [ ]:
show(tables.conclusion_noise("adaptationShare"))

## 3 · El peldaño del peso por confianza

La mitad que las dos tablas de arriba no aíslan, y la que el destino contaminado
pone en juego. El destino entrena **sin etiquetas** — `pseudolabel` (Ec. 22) y
`confidences` (Ec. 24) —, así que contaminar sus bolsas no es ruido de etiqueta:
corrompe la condicional a la que el término de adaptación se alinea y envenena
los pseudo-rótulos de los que sale la confianza.

`D` contra `C`, y `F` y `G` contra `E`, difieren **exactamente** en ese peso. Con
material limpio ese peldaño casi no tiene nada que separarlo; es bajo ruido donde
el peso tiene algo que hacer, y por eso se lee acá y no en la campaña limpia.

In [ ]:
show(tables.objective("noise.share"))

In [ ]:
show(tables.conclusion_weighting_under_noise("targetAccuracy"))

## 4 · El registro

Lo que este cuaderno midió, escrito donde la declaración dice que vive. El
`records` del paquete nombra `Results/Noise/degradation.json` justamente para que
esto no aparezca después como un experimento que nadie contabilizó.

In [ ]:
destino = config.PRODUCT / "Results" / "Noise"
destino.mkdir(parents=True, exist_ok=True)
registro = {
    "transfer": TRANSFERENCIA,
    "levels": config.NOISE_LEVELS,
    "ran": corridos,
    "missing": faltan,
    "curves": {m: contamination.curve(m) for m in ("targetAccuracy", "adaptationShare")},
    "degradation": {m: contamination.degradation(m) for m in ("targetAccuracy", "adaptationShare")},
    "revision": config.REVISION,
}
(destino / "degradation.json").write_text(json.dumps(registro, indent=2), encoding="utf-8")
print(f"escrito: {destino / 'degradation.json'}")